# CELL 0: Project Overview & Configuration
"""
Project Title: 📚 Smart Auto-Lecture Note Taker 📚
Track: Track B > Multimodal Application

Description:
This project implements an end-to-end multimodal pipeline that transforms raw
lecture video files into professional, structured study notes.

Technical Pipeline:
1. Audio Extraction: Processes video files to isolate clear audio tracks.
2. Speech-to-Text: Transcribes speech using the robust OpenAI Whisper model.
3. LLM Reasoning: Structures the content into study notes using Qwen2.5-1.5B (4-bit quantized).

Performance Note (T4 GPU):
The pipeline is optimized for academic efficiency. Processing times scale linearly:
- Every 5 minutes of video take approximately 1 minute of processing time.


Developer Information:
- GitHub: [https://github.com/0fb7]
- X (Twitter): [https://x.com/prog7_]

This project directly reinforces the 'Multimodal Data' and 'Software Development'
"""

In [ ]:
# CELL 1: Install required libraries
# ADDITION: 'moviepy' is added here to support your idea of extracting audio from video files.
!pip -q install -U "transformers>=4.56,<6" accelerate "datasets>=4,<6" soundfile librosa moviepy gradio "bitsandbytes>=0.46.1"

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 11.5/11.5 MB 42.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 555.1/555.1 kB 20.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 129.9/129.9 kB 3.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 31.0/31.0 MB 27.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 60.7/60.7 MB 9.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 50.1/50.1 MB 12.8 MB/s eta 0:00:00


In [ ]:
# CELL 2: Import libraries and suppress warnings
import warnings
warnings.filterwarnings("ignore")

import torch
import gradio as gr

# ADDITION: VideoFileClip from moviepy is imported to handle the video upload idea.
from moviepy import VideoFileClip
from transformers import pipeline, AutoTokenizer, AutoModelForCausalLM, BitsAndBytesConfig

In [ ]:
# CELL 3: Setup Device and Load AI Models (Whisper & Qwen)
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
dtype = torch.float16 if DEVICE == "cuda" else torch.float32

# 1. Load Whisper for Speech-to-Text (using 'small' for better lecture accuracy)
print("Loading Whisper model...")
asr_pipeline = pipeline(
    "automatic-speech-recognition",
    model="openai/whisper-small",
    device=0 if DEVICE == "cuda" else -1,
    dtype=dtype
)

# 2. Load Qwen LLM for note structuring (using 4-bit quantization to save memory)
print("Loading Qwen LLM model...")
bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_compute_dtype=torch.float16,
    bnb_4bit_use_double_quant=True,
)

llm_id = "Qwen/Qwen2.5-1.5B-Instruct"
tokenizer = AutoTokenizer.from_pretrained(llm_id)
llm = AutoModelForCausalLM.from_pretrained(
    llm_id,
    quantization_config=bnb_config,
    device_map="cuda"
)

Loading Whisper model...


config.json:   0%|          | 0.00/1.97k [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/967M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/479 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/3.87k [00:00<?, ?B/s]

...

merges.txt:   0%|          | 0.00/1.67M [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/7.03M [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/3.09G [00:00<?, ?B/s]

Loading weights:   0%|          | 0/338 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/242 [00:00<?, ?B/s]

In [ ]:
# CELL 4: Video to Audio Extraction Function
# ADDITION: This entire function is added to support your idea of processing video files.
# Whisper cannot read videos directly, so this function extracts the audio track
# and saves it as a 16kHz .wav file (the optimal format for Whisper).

def extract_audio_from_video(video_path):
    audio_path = "extracted_audio.wav"
    # Extract audio and save it at 16000Hz
    clip = VideoFileClip(video_path)
    clip.audio.write_audiofile(audio_path, fps=16000, logger=None)
    clip.close()

    return audio_path

In [ ]:
# CELL 5: LLM Generation Function (Applying your custom prompt)
def generate_study_notes(transcript):
    # Your comprehensive and professional prompt
    user_prompt = """I will upload an audio recording. Treat it as a lecture or educational lesson, not as a simple transcription.

Your task is to transform the recording into a well-structured, professional study note.

Requirements:
1. Listen to the entire recording before generating the output.
2. Do NOT produce a verbatim transcript. Instead, rewrite the content into a clear, organized lesson while preserving the original meaning.
3. Organize the content into logical sections with descriptive main headings and subheadings.
4. Use clear bullet points under each section.
5. Explain concepts only when the speaker already provides enough context. Do not introduce external information or assumptions.
6. Extract every important piece of information mentioned, including:
   - Definitions
   - Key concepts
   - Rules, formulas, or equations
   - Procedures and step-by-step processes
   - Conditions and exceptions
   - Examples
   - Tips and best practices
   - Common mistakes or misconceptions
   - Questions asked by the instructor and their answers
   - Anything explicitly emphasized as important or likely to appear on an exam
7. Remove filler words, repetitions, pauses, and conversational language unless they affect the meaning.
8. If the lecturer changes topics, create a new section with an appropriate title.
9. Keep the original order of ideas unless reorganizing slightly improves clarity.
10. Write in a clean, professional style suitable for studying.

At the end, include the following sections:

# Executive Summary
# Key Takeaways
# Definitions
# Rules / Formulas / Equations
# Step-by-Step Procedures
# Common Mistakes
# Important Notes
# Review Questions
# Final Cheat Sheet

Important Instructions:
- Never invent information that was not stated or clearly implied.
- If any word or sentence is unintelligible, write [Unclear].
- Preserve all technical terms accurately.
- Use Markdown formatting.
"""

    # Combine the prompt with the extracted transcript
    full_input = f"{user_prompt}\n\nHere is the Transcript:\n{transcript}"

    messages = [{"role": "user", "content": full_input}]
    inputs_txt = tokenizer.apply_chat_template(
        messages,
        add_generation_prompt=True,
        return_tensors="pt",
        return_dict=False
    ).to(DEVICE)

    # max_new_tokens is set to 3000 to accommodate the long output required by your prompt
    out = llm.generate(
        inputs_txt,
        max_new_tokens=3000,
        do_sample=True,
        temperature=0.7,
        top_p=0.95,
        pad_token_id=tokenizer.eos_token_id
    )

    notes = tokenizer.decode(out[0][inputs_txt.shape[1]:], skip_special_tokens=True)
    return notes

In [ ]:
# CELL 6: Main Pipeline linking all components
def process_lecture_video(video_file):
    if video_file is None:
        return "Please upload a video.", "No notes generated."

    try:
        # Step 1: Extract audio from video
        # ADDITION: Calling the newly added function here to prepare the file for Whisper.
        audio_path = extract_audio_from_video(video_file)

        # Step 2: Transcribe the audio
        # Note: chunk_length_s=30 is crucial for processing long lecture videos.
        transcript = asr_pipeline(audio_path, chunk_length_s=30)["text"].strip()

        # Step 3: Generate structured notes using Qwen LLM
        notes = generate_study_notes(transcript)

        return transcript, notes
    except Exception as e:
        return f"Error processing video: {str(e)}", ""

In [ ]:
# CELL 7: Gradio User Interface and App Launch
# ADDITION: gr.Video is used as the input component to support the user uploading video lectures.

demo = gr.Interface(
    fn=process_lecture_video,
    inputs=gr.Video(label="Upload Lecture Video (MP4, AVI, etc.)"),
    outputs=[
        gr.Textbox(label="Raw Transcript (Whisper)", lines=5),
        gr.Markdown(label="Structured Study Notes (Qwen)")
    ],
    title="📚 Smart Auto-Lecture Note Taker 📚",
    description="Upload a lecture video. The AI will extract the audio, transcribe it, and transform it into a highly structured, professional study guide."
)

# Launch the app and generate a shareable link
demo.launch(share=True)

Colab notebook detected. To show errors in colab notebook, set debug=True in launch()
* Running on public URL: https://e8d5baca98981e74a0.gradio.live

This share link is temporary and will last for up to 1 week (best effort). For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)


In [1]:
# CELL 7: Generate files for Hugging Face Space deployment
# This cell automatically creates 'app.py' and 'requirements.txt'
# based on your current working implementation.

# 1. Create the 'app.py' file
app_py_content = '''import gradio as gr
import torch
# Updated import for newer moviepy versions
from moviepy import VideoFileClip
from transformers import pipeline, AutoTokenizer, AutoModelForCausalLM, BitsAndBytesConfig

# Load models
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
dtype = torch.float16 if torch.cuda.is_available() else torch.float32

asr_pipeline = pipeline("automatic-speech-recognition", model="openai/whisper-small", device=0 if torch.cuda.is_available() else -1, dtype=dtype)

bnb_config = BitsAndBytesConfig(load_in_4bit=True, bnb_4bit_quant_type="nf4", bnb_4bit_compute_dtype=torch.float16, bnb_4bit_use_double_quant=True)
llm = AutoModelForCausalLM.from_pretrained("Qwen/Qwen2.5-1.5B-Instruct", quantization_config=bnb_config, device_map="cuda")
tokenizer = AutoTokenizer.from_pretrained("Qwen/Qwen2.5-1.5B-Instruct")

def extract_audio(video_path):
    audio_path = "extracted_audio.wav"
    clip = VideoFileClip(video_path)
    clip.audio.write_audiofile(audio_path, fps=16000, logger=None)
    clip.close()
    return audio_path

def generate_notes(video_file):
    audio = extract_audio(video_file)
    transcript = asr_pipeline(audio, chunk_length_s=30)["text"]

    user_prompt = """I will upload an audio recording. Treat it as a lecture or educational lesson, not as a simple transcription.
Your task is to transform the recording into a well-structured, professional study note.

Requirements:
1. Listen to the entire recording before generating the output.
2. Do NOT produce a verbatim transcript. Instead, rewrite the content into a clear, organized lesson while preserving the original meaning.
3. Organize the content into logical sections with descriptive main headings and subheadings.
4. Use clear bullet points under each section.
5. Explain concepts only when the speaker already provides enough context. Do not introduce external information or assumptions.
6. Extract every important piece of information mentioned, including:
   - Definitions, Key concepts, Rules, formulas, or equations, Procedures, Exceptions, Examples, Tips, Mistakes, Questions & Answers, Exam emphasis.
7. Remove filler words, repetitions, and pauses.
8. If the lecturer changes topics, create a new section.
9. Keep the original order of ideas.
10. Write in a clean, professional style.

At the end, include the following sections:
# Executive Summary
# Key Takeaways
# Definitions
# Rules / Formulas / Equations
# Step-by-Step Procedures
# Common Mistakes
# Important Notes
# Review Questions
# Final Cheat Sheet

Important Instructions:
- Never invent information that was not stated.
- If any word or sentence is unintelligible, write [Unclear].
- Use Markdown formatting.
"""
    prompt = user_prompt + "\\n\\nHere is the Transcript:\\n" + transcript
    messages = [{"role": "user", "content": prompt}]
    inputs = tokenizer.apply_chat_template(messages, add_generation_prompt=True, return_tensors="pt").to("cuda")
    out = llm.generate(inputs, max_new_tokens=2000, pad_token_id=tokenizer.eos_token_id)
    return tokenizer.decode(out[0][inputs.shape[1]:], skip_special_tokens=True)

APP_TITLE = "Smart Auto-Lecture Note Taker Developed by 👨‍💻Fares Basaleh👨‍💻"
demo = gr.Interface(fn=generate_notes, inputs=gr.Video(), outputs="markdown",title=APP_TITLE)
demo.launch()
'''

# 2. Create the 'requirements.txt' file
requirements_content = '''transformers>=4.56
accelerate
bitsandbytes
gradio
moviepy==1.0.3
torch
pillow
soundfile
librosa
sentencepiece
'''

# Write files
with open("app.py", "w") as f:
    f.write(app_py_content)
with open("requirements.txt", "w") as f:
    f.write(requirements_content)

print("Files 'app.py' and 'requirements.txt' created successfully!")

Files 'app.py' and 'requirements.txt' created successfully!
